# 02. Appropriations Intent: Node Classification

Classify BillNode text by appropriations intent (appropriation, restriction, transfer, rescission, cap).
Then build an enriched financial summary that adds labels to the diff's dollar-amount changes.


In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")

from bill_tree import normalize_bill

## Load and explore sample bill

In [4]:
# Load bill
xml_path = Path("../bills/118-hr-4366/1_reported-in-house.xml")
tree = normalize_bill(xml_path)

In [ ]:
# inspect structure
print(f"Total nodes: {len(tree.nodes)}")
print(tree.nodes[0])

dollar_nodes = [node for node in tree.nodes if re.search(r"\$[\d,]+", node.body_text or "")]
print(f"\nNodes with dollar amounts: {len(dollar_nodes)}")

for node in dollar_nodes[:3]:
    print("---")
    print(node.display_path)
    print(node.body_text[:300])

Total nodes: 205
BillNode(match_path=('front matter', 'masthead'), display_path=(), tag='front-matter', element_id='front-matter-masthead', header_text='', body_text='118th CONGRESS\n1st Session\nH. R. 4366\nA BILL', section_number='', division_label='', display_text='')

Nodes with dollar amounts: 67
---
('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, army')
For acquisition, construction, installation, and equipment of temporary or permanent public works, military installations, facilities, and real property for the Army as currently authorized by law, including personnel in the Army Corps of Engineers and other personal services necessary for the purpo
---
('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, navy and marine corps')
For acquisition, construction, installation, and equipment of temporary or permanent public works, naval installations, facilities, and real property for the Navy and Marine Corps as currently authorized by law, including personnel in the Na

In [8]:
provided_nodes = [
    node
    for node in tree.nodes
    if re.search(r"Provided.{0,200}\$[\d,]+", node.body_text or "")
    and not re.search(r"^\s*For .{0,500}\$[\d,]+.*to remain available", node.body_text or "")
]

for node in provided_nodes[:5]:
    print("---")
    print(node.display_path)
    print(node.body_text[:400])
    print()

---
('TITLE I—DEPARTMENT OF DEFENSE', 'Administrative provisions', 'sec. 119')
Notwithstanding any other provision of law, funds made available in this title for operation and maintenance of family housing shall be the exclusive source of funds for repair and maintenance of all family housing units, including general or flag officer quarters: Provided, That not more than $15,000 per unit may be spent annually for the maintenance and repair of any general or flag officer quar

---
('TITLE II', 'DEPARTMENT OF VETERANS AFFAIRS', 'Veterans benefits administration', 'COMPENSATION AND PENSIONS')
For the payment of compensation benefits to or on behalf of veterans and a pilot program for disability examinations as authorized by section 107 and chapters 11, 13, 18, 51, 53, 55, and 61 of title 38, United States Code; pension benefits to or on behalf of veterans as authorized by chapters 15, 51, 53, 55, and 61 of title 38, United States Code; and burial benefits, the Reinstated Entitlement Pr

-

In [9]:
cap_pattern = re.compile(r"not (?:more|to exceed) \$[\d,]+", re.IGNORECASE)
approp_pattern = re.compile(r"^For .{0,500}\$[\d,]+.*to remain available", re.DOTALL)

for node in dollar_nodes[:20]:
    has_cap = bool(cap_pattern.search(node.body_text or ""))
    has_approp = bool(approp_pattern.search(node.body_text or ""))
    print(f"cap={has_cap} approp={has_approp} | {node.display_path[-1] if node.display_path else ''}")

cap=True approp=True | Military construction, army
cap=True approp=True | Military construction, navy and marine corps
cap=True approp=True | Military construction, air force
cap=True approp=True | Military construction, defense-Wide
cap=True approp=True | Military construction, army national guard
cap=True approp=True | Military construction, air national guard
cap=True approp=True | Military construction, army reserve
cap=True approp=True | Military construction, navy reserve
cap=True approp=True | Military construction, air force reserve
cap=False approp=True | Security investment program
cap=False approp=True | Department of defense base closure account
cap=False approp=True | Family housing construction, army
cap=False approp=False | Family housing operation and maintenance, army
cap=False approp=True | Family housing construction, navy and marine corps
cap=False approp=False | Family housing operation and maintenance, navy and marine corps
cap=False approp=True | Family housing c

In [11]:
mystery_nodes = [
    node
    for node in dollar_nodes
    if not cap_pattern.search(node.body_text or "") and not approp_pattern.search(node.body_text or "")
]

for node in mystery_nodes:
    print("---")
    print(node.display_path)
    print(node.body_text)
    print()

---
('TITLE I—DEPARTMENT OF DEFENSE', 'Family housing operation and maintenance, army')
For expenses of family housing for the Army for operation and maintenance, including debt payment, leasing, minor construction, principal and interest charges, and insurance premiums, as authorized by law, $395,485,000.

---
('TITLE I—DEPARTMENT OF DEFENSE', 'Family housing operation and maintenance, navy and marine corps')
For expenses of family housing for the Navy and Marine Corps for operation and maintenance, including debt payment, leasing, minor construction, principal and interest charges, and insurance premiums, as authorized by law, $373,854,000.

---
('TITLE I—DEPARTMENT OF DEFENSE', 'Family housing operation and maintenance, air force')
For expenses of family housing for the Air Force for operation and maintenance, including debt payment, leasing, minor construction, principal and interest charges, and insurance premiums, as authorized by law, $324,386,000.

---
('TITLE I—DEPARTMENT OF D

## Build rule-based classifier

In [29]:
APPROP = re.compile(r"^\s*For\b", re.IGNORECASE | re.DOTALL)
RESTRICT = re.compile(r"^\s*None of the funds", re.IGNORECASE)
TRANSFER = re.compile(r"^\s*Of (?:the )?amounts", re.IGNORECASE)
RESCISSION = re.compile(r"is hereby rescinded", re.IGNORECASE)
CAP = re.compile(r"not (?:more than|to exceed)\s+\$[\d,]+", re.IGNORECASE)
DOLLAR = re.compile(r"\$([\d,]+(?:\.\d+)?)")


def classify_text(text):
    if not text:
        return None
    if RESTRICT.match(text):
        return "restriction"
    if TRANSFER.match(text):
        return "transfer"
    if APPROP.match(text):
        return "rescission" if RESCISSION.search(text) else "appropriation"
    if RESCISSION.search(text):
        return "rescission"
    if CAP.search(text):
        return "cap"
    return "unknown"


def primary_amount(text):
    if not text:
        return None
    pre_provided = text.split("Provided")[0]
    m = DOLLAR.search(pre_provided)
    return float(m.group(1).replace(",", "")) if m else None

### Single-bill classification

Load one version and classify every dollar-amount node.
Review `unknown` rows to find gaps in the patterns.

In [30]:
tree = normalize_bill(Path("../bills/118-hr-4366/1_reported-in-house.xml"))

dollar_nodes = [n for n in tree.nodes if DOLLAR.search(n.body_text or "")]

rows = [
    {
        "label": classify_text(n.body_text),
        "amount": primary_amount(n.body_text),
        "path": " > ".join(n.display_path[-2:]) if n.display_path else "",
        "preview": (n.body_text or "")[:150],
    }
    for n in dollar_nodes
]

df_nodes = pd.DataFrame(rows)
print(df_nodes["label"].value_counts())
df_nodes

label
appropriation    51
restriction       5
unknown           5
transfer          4
cap               2
Name: count, dtype: int64


,label,amount,path,preview
0,appropriation,1.517455e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
1,appropriation,4.477961e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
2,appropriation,2.439614e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
3,appropriation,2.651047e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
4,appropriation,3.692610e+08,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For construction, acquisition, expansion, reha..."
...,...,...,...,...
62,appropriation,4.720000e+07,United states court of appeals for veterans cl...,For necessary expenses for the operation of th...
63,appropriation,2.000000e+03,"Cemeterial expenses, army > SALARIES AND EXPENSES","For necessary expenses for maintenance, operat..."
64,appropriation,8.860000e+07,"Cemeterial expenses, army > CONSTRUCTION",For necessary expenses for planning and design...
65,appropriation,7.700000e+07,Armed forces retirement home > TRUST FUND,For expenses necessary for the Armed Forces Re...


In [ ]:
# Review unknowns
df_nodes[df_nodes["label"] == "unknown"][["path", "preview"]]

,path,preview
24,Administrative provisions > sec. 113,The Secretary of Defense shall inform the appr...
56,Administrative provisions > sec. 223,The Secretary of Veterans Affairs shall notify...
57,Administrative provisions > sec. 227,The Secretary of Veterans Affairs shall provid...
58,Administrative provisions > sec. 230,The Secretary of Veterans Affairs may not repr...
66,GENERAL PROVISIONS > sec. 418,$0.


In [33]:
node = dollar_nodes[66]
print("display_path:", node.display_path)
print("header_text: ", node.header_text)
print("section_number:", node.section_number)
print("body_text:", node.body_text)

display_path: ('TITLE IV', 'GENERAL PROVISIONS', 'sec. 418')
header_text:  
section_number: Sec. 418
body_text: $0.
